# Linear Regression — Retail Sales Forecasting

This notebook builds **Linear Regression** models (OLS, Ridge, and Lasso) to forecast hourly retail sales.  
It uses the same data pipeline (hourly aggregation, 95th-percentile capping, 168-hour holdout) as the Random Forest, Gradient Boosting, and XGBoost models for a fair comparison.

Linear models serve as interpretable baselines — if tree-based models only marginally beat them, the relationship may be mostly linear.

---

## 1 · Imports & Setup

In [ ]:
# ── Core ────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ──────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11
})

# ── Scikit-learn ───────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ── Persistence ────────────────────────────────────
import joblib, os

print('All imports loaded successfully ✓')

---
## 2 · Data Loading

In [ ]:
DATA_PATH = os.path.join('..', 'Forecasting', 'cleaned_online_retail.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['InvoiceDate'])
print(f'Dataset shape : {df.shape}')
print(f'Date range    : {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
df.head()

---
## 3 · Feature Engineering

### 3.1 — TotalSales & Hourly Aggregation

In [ ]:
# Revenue per line item
df['TotalSales'] = df['UnitPrice'] * df['Quantity']

# Resample to hourly frequency
hourly_df = (
    df
    .set_index('InvoiceDate')['TotalSales']
    .resample('h')
    .sum()
    .reset_index()
)

# Cap extreme values at the 95th percentile
cap = hourly_df['TotalSales'].quantile(0.95)
hourly_df['CappedSales'] = np.where(
    hourly_df['TotalSales'] > cap, cap, hourly_df['TotalSales']
)

print(f'Hourly records : {len(hourly_df)}')
print(f'95th-pctl cap  : £{cap:,.2f}')
hourly_df.head()

### 3.2 — Prepare ML DataFrame & Engineer Features

In [ ]:
ml_df = hourly_df[['InvoiceDate', 'CappedSales']].copy()
ml_df.columns = ['ds', 'y']
ml_df.set_index('ds', inplace=True)

# ── Calendar features ──
ml_df['hour']         = ml_df.index.hour
ml_df['day_of_week']  = ml_df.index.dayofweek
ml_df['day_of_month'] = ml_df.index.day
ml_df['month']        = ml_df.index.month
ml_df['week_of_year'] = ml_df.index.isocalendar().week.astype(int)
ml_df['quarter']      = ml_df.index.quarter
ml_df['is_weekend']   = (ml_df.index.dayofweek >= 5).astype(int)

# ── Interaction ──
ml_df['hour_weekend'] = ml_df['hour'] * ml_df['is_weekend']

# ── Lag features ──
ml_df['lag_1']   = ml_df['y'].shift(1)
ml_df['lag_24']  = ml_df['y'].shift(24)
ml_df['lag_168'] = ml_df['y'].shift(168)

# ── Rolling features ──
ml_df['rolling_mean_24']  = ml_df['y'].shift(1).rolling(window=24).mean()
ml_df['rolling_std_24']   = ml_df['y'].shift(1).rolling(window=24).std()
ml_df['rolling_mean_168'] = ml_df['y'].shift(1).rolling(window=168).mean()

# ── Drop NaN rows ──
rows_before = len(ml_df)
ml_df.dropna(inplace=True)
print(f'Rows dropped: {rows_before - len(ml_df)}')
print(f'Final shape : {ml_df.shape}  ({ml_df.shape[1]-1} features)')
ml_df.head()

---
## 4 · Train / Test Split & Feature Scaling

Linear models require **feature scaling** for optimal performance (especially Ridge and Lasso).

In [ ]:
HOLDOUT_HOURS = 168

train = ml_df.iloc[:-HOLDOUT_HOURS]
test  = ml_df.iloc[-HOLDOUT_HOURS:]

feature_cols = [c for c in ml_df.columns if c != 'y']

X_train_raw, y_train = train[feature_cols], train['y']
X_test_raw,  y_test  = test[feature_cols],  test['y']

# ── Standardize features ──
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=feature_cols, index=X_train_raw.index)
X_test  = pd.DataFrame(scaler.transform(X_test_raw), columns=feature_cols, index=X_test_raw.index)

print(f'Training : {X_train.shape[0]:,} samples  ({train.index.min().date()} → {train.index.max().date()})')
print(f'Test     : {X_test.shape[0]:,} samples  ({test.index.min().date()} → {test.index.max().date()})')
print(f'Features : {X_train.shape[1]}  (standardized)')

---
## 5 · Ordinary Least Squares (OLS) Linear Regression

In [ ]:
lr_ols = LinearRegression()
lr_ols.fit(X_train, y_train)
y_pred_ols = lr_ols.predict(X_test)

mae_ols  = mean_absolute_error(y_test, y_pred_ols)
rmse_ols = np.sqrt(mean_squared_error(y_test, y_pred_ols))
r2_ols   = r2_score(y_test, y_pred_ols)

print('── OLS Linear Regression ───────────────────')
print(f'  MAE  : £{mae_ols:,.2f}')
print(f'  RMSE : £{rmse_ols:,.2f}')
print(f'  R²   : {r2_ols:.4f}')

---
## 6 · Ridge Regression (L2 Regularization)

Use **GridSearchCV** to find the optimal regularization strength alpha.

In [ ]:
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 50, 100, 500, 1000]}

ridge_search = GridSearchCV(
    Ridge(),
    ridge_params,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1
)
ridge_search.fit(X_train, y_train)

ridge_best = ridge_search.best_estimator_
y_pred_ridge = ridge_best.predict(X_test)

mae_ridge  = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_ridge   = r2_score(y_test, y_pred_ridge)

print('── Ridge Regression ────────────────────────')
print(f'  Best alpha : {ridge_search.best_params_["alpha"]}')
print(f'  MAE  : £{mae_ridge:,.2f}')
print(f'  RMSE : £{rmse_ridge:,.2f}')
print(f'  R²   : {r2_ridge:.4f}')

---
## 7 · Lasso Regression (L1 Regularization)

In [ ]:
lasso_params = {'alpha': [0.01, 0.1, 1, 10, 50, 100, 500, 1000]}

lasso_search = GridSearchCV(
    Lasso(max_iter=10000),
    lasso_params,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1
)
lasso_search.fit(X_train, y_train)

lasso_best = lasso_search.best_estimator_
y_pred_lasso = lasso_best.predict(X_test)

mae_lasso  = mean_absolute_error(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
r2_lasso   = r2_score(y_test, y_pred_lasso)

print('── Lasso Regression ────────────────────────')
print(f'  Best alpha : {lasso_search.best_params_["alpha"]}')
print(f'  MAE  : £{mae_lasso:,.2f}')
print(f'  RMSE : £{rmse_lasso:,.2f}')
print(f'  R²   : {r2_lasso:.4f}')

---
## 8 · Best Model Selection

Compare OLS, Ridge, and Lasso — select the model with the lowest MAE.

In [ ]:
linear_results = {
    'OLS':   {'model': lr_ols,     'preds': y_pred_ols,   'mae': mae_ols,   'rmse': rmse_ols,   'r2': r2_ols},
    'Ridge': {'model': ridge_best, 'preds': y_pred_ridge, 'mae': mae_ridge, 'rmse': rmse_ridge, 'r2': r2_ridge},
    'Lasso': {'model': lasso_best, 'preds': y_pred_lasso, 'mae': mae_lasso, 'rmse': rmse_lasso, 'r2': r2_lasso},
}

best_name = min(linear_results, key=lambda k: linear_results[k]['mae'])
best_info = linear_results[best_name]
lr_best      = best_info['model']
y_pred_best  = best_info['preds']
best_label   = best_name

print(f'★ {best_name} selected as best linear model (lowest MAE)\n')

linear_comparison = pd.DataFrame({
    'Model':    ['OLS', 'Ridge', 'Lasso'],
    'MAE (£)':  [round(mae_ols, 2), round(mae_ridge, 2), round(mae_lasso, 2)],
    'RMSE (£)': [round(rmse_ols, 2), round(rmse_ridge, 2), round(rmse_lasso, 2)],
    'R²':       [round(r2_ols, 4), round(r2_ridge, 4), round(r2_lasso, 4)],
})
print('── Linear Model Comparison ─────────────────')
print(linear_comparison.to_string(index=False))

---
## 9 · Cross-Model Comparison

Load Random Forest and Gradient Boosting models for side-by-side evaluation.

In [ ]:
# Start with linear model results
comparison_data = {
    'Model':    [f'Linear Regression ({best_name})'],
    'MAE (£)':  [round(best_info['mae'], 2)],
    'RMSE (£)': [round(best_info['rmse'], 2)],
    'R²':       [round(best_info['r2'], 4)],
}

# ── Load RF model ──
rf_path = 'rf_sales_model.pkl'
if os.path.exists(rf_path):
    try:
        rf_model = joblib.load(rf_path)
        # RF needs unscaled features
        y_pred_rf = rf_model.predict(X_test_raw)
        comparison_data['Model'].append('Random Forest (Best)')
        comparison_data['MAE (£)'].append(round(mean_absolute_error(y_test, y_pred_rf), 2))
        comparison_data['RMSE (£)'].append(round(np.sqrt(mean_squared_error(y_test, y_pred_rf)), 2))
        comparison_data['R²'].append(round(r2_score(y_test, y_pred_rf), 4))
        print('Random Forest model loaded ✓')
    except Exception as e:
        print(f'Could not load RF model: {e}')

# ── Load GB model ──
gb_path = 'gb_sales_model.pkl'
if os.path.exists(gb_path):
    try:
        gb_model = joblib.load(gb_path)
        # GB needs unscaled features
        y_pred_gb = gb_model.predict(X_test_raw)
        comparison_data['Model'].append('Gradient Boosting (Best)')
        comparison_data['MAE (£)'].append(round(mean_absolute_error(y_test, y_pred_gb), 2))
        comparison_data['RMSE (£)'].append(round(np.sqrt(mean_squared_error(y_test, y_pred_gb)), 2))
        comparison_data['R²'].append(round(r2_score(y_test, y_pred_gb), 4))
        print('Gradient Boosting model loaded ✓')
    except Exception as e:
        print(f'Could not load GB model: {e}')

# ── Load XGBoost model ──
xgb_path = os.path.join('..', 'Forecasting', 'xgb_sales_forecast_model.pkl')
if os.path.exists(xgb_path):
    try:
        import xgboost as xgb
        model_xgb = joblib.load(xgb_path)
        X_test_xgb = X_test_raw[['hour', 'day_of_week', 'month', 'is_weekend',
                                  'lag_1', 'lag_24', 'lag_168', 'rolling_mean_24']].copy()
        X_test_xgb.rename(columns={'day_of_week': 'dayofweek'}, inplace=True)
        y_pred_xgb = model_xgb.predict(X_test_xgb)
        comparison_data['Model'].append('XGBoost (Friend\'s Model)')
        comparison_data['MAE (£)'].append(round(mean_absolute_error(y_test, y_pred_xgb), 2))
        comparison_data['RMSE (£)'].append(round(np.sqrt(mean_squared_error(y_test, y_pred_xgb)), 2))
        comparison_data['R²'].append(round(r2_score(y_test, y_pred_xgb), 4))
        print('XGBoost model loaded ✓')
    except Exception as e:
        print(f'Could not load XGBoost model: {e}')

comparison_df = pd.DataFrame(comparison_data)
print('\n── Full Model Comparison (168-hour holdout) ─')
print(comparison_df.to_string(index=False))

---
## 10 · Visualizations

### 10.1 — Actual vs Predicted

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(test.index, y_test, label='Actual', color='#2563EB', linewidth=1.5)
ax.plot(test.index, y_pred_best, label=f'Linear Regression ({best_label})', color='#EC4899',
        linewidth=1.5, linestyle='--')
ax.set_title(f'Hourly Sales — Actual vs Linear Regression ({best_label})',
             fontsize=15, fontweight='bold')
ax.set_xlabel('Date & Hour')
ax.set_ylabel('Sales (£)')
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('lr_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → lr_actual_vs_predicted.png')

### 10.2 — Residual Analysis

In [ ]:
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(range(len(residuals)), residuals,
            color=np.where(residuals >= 0, '#10B981', '#EF4444'), alpha=0.7, width=1.0)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Residuals Over Time', fontweight='bold')
axes[0].set_xlabel('Test Hour Index')
axes[0].set_ylabel('Residual (£)')

axes[1].hist(residuals, bins=30, color='#EC4899', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.2)
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (£)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('lr_residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → lr_residual_analysis.png')

### 10.3 — Feature Coefficients

Unlike tree models, linear models have **coefficients** that show feature direction and magnitude.

In [ ]:
coefficients = pd.Series(lr_best.coef_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#EF4444' if c < 0 else '#10B981' for c in coefficients]
coefficients.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title(f'{best_label} Regression — Feature Coefficients (Standardized)',
             fontsize=15, fontweight='bold')
ax.set_xlabel('Coefficient Value')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('lr_feature_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → lr_feature_coefficients.png')

### 10.4 — Actual vs Predicted Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred_best, alpha=0.5, s=30, color='#EC4899', edgecolors='white', linewidth=0.3)

lims = [min(0, y_pred_best.min() * 1.05), max(y_test.max(), y_pred_best.max()) * 1.05]
ax.plot(lims, lims, '--', color='#EF4444', linewidth=1.5, label='Perfect Prediction')

ax.set_title(f'Actual vs Predicted Sales ({best_label})', fontsize=15, fontweight='bold')
ax.set_xlabel('Actual Sales (£)')
ax.set_ylabel('Predicted Sales (£)')
ax.legend()
ax.set_xlim(lims)
ax.set_ylim(lims)
plt.tight_layout()
plt.savefig('lr_scatter_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → lr_scatter_actual_vs_predicted.png')

### 10.5 — Lasso Feature Selection

Lasso drives unimportant feature coefficients to zero — useful for feature selection.

In [ ]:
lasso_coefs = pd.Series(lasso_best.coef_, index=feature_cols)
zero_features = lasso_coefs[lasso_coefs == 0].index.tolist()
nonzero_features = lasso_coefs[lasso_coefs != 0].index.tolist()

print(f'Lasso kept {len(nonzero_features)}/{len(feature_cols)} features (alpha={lasso_best.alpha})')
print(f'\nKept features     : {nonzero_features}')
print(f'Dropped features  : {zero_features if zero_features else "None (all features kept)"}')

---
## 11 · Model Persistence

In [ ]:
# Save the best linear model and the scaler
MODEL_PATH  = 'lr_sales_model.pkl'
SCALER_PATH = 'lr_scaler.pkl'

joblib.dump(lr_best, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

print(f'Model saved  → {MODEL_PATH}  (variant: {best_label})')
print(f'Scaler saved → {SCALER_PATH}')
print(f'Model size   : {os.path.getsize(MODEL_PATH) / 1024:.1f} KB')

### Sanity check — reload & predict

In [ ]:
lr_loaded     = joblib.load(MODEL_PATH)
scaler_loaded = joblib.load(SCALER_PATH)
X_test_check  = scaler_loaded.transform(X_test_raw)
y_check       = lr_loaded.predict(X_test_check)
assert np.allclose(y_check, y_pred_best), 'Mismatch after reload!'
print('Reload sanity check passed ✓')

---
## Summary

| Step | Status |
|------|--------|
| Data loading & cleaning | ✓ |
| Feature engineering (14 features) | ✓ |
| Feature scaling (StandardScaler) | ✓ |
| OLS Linear Regression | ✓ |
| Ridge Regression (L2, GridSearchCV) | ✓ |
| Lasso Regression (L1, GridSearchCV) | ✓ |
| Best model selection | ✓ |
| Cross-model comparison (RF, GB, XGBoost) | ✓ |
| Visualizations (5 charts) | ✓ |
| Lasso feature selection analysis | ✓ |
| Model + scaler persistence (.pkl) | ✓ |